# 06a - Interpretabilitas Model
Materi Modul 6: interpretabilitas model, contoh model yang dapat/tidak dapat diinterpretasikan sendiri, penjelasan model-agnostik, model pengganti (surrogate).

- **Dapat diinterpretasikan sendiri**: regresi logistik (koefisien), pohon keputusan kecil
- **Black box**: random forest, gradient boosting, SVM kernel
- **Model-agnostik**: cara menjelaskan model apa pun (permutation importance, partial dependence)
- **Model pengganti (surrogate)**: model sederhana yang meniru prediksi model black box

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import accuracy_score

df = pd.read_csv('../datasets/breast_cancer.csv')
X, y = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 1. Model yang dapat diinterpretasikan sendiri

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
koef = pd.Series(lr[-1].coef_[0], index=X.columns).sort_values()
print('Akurasi regresi logistik:', round(lr.score(X_test, y_test), 4))
print(koef.tail(5).round(2))

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print('\nAkurasi pohon kecil:', round(tree.score(X_test, y_test), 4))
print(export_text(tree, feature_names=list(X.columns)))

## 2. Model black box

In [ ]:
black = GradientBoostingClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
print('Akurasi Gradient Boosting (black box):', round(black.score(X_test, y_test), 4))

## 3. Penjelasan model-agnostik: permutation importance
Acak satu fitur di data test. Kalau akurasi turun banyak, fitur itu penting. Bisa dipakai di model apa saja.

In [ ]:
pi = permutation_importance(black, X_test, y_test, n_repeats=10, random_state=42)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values()

imp.tail(10).plot(kind='barh', xerr=pd.Series(pi.importances_std, index=X.columns)[imp.tail(10).index], figsize=(7, 4))
plt.xlabel('Penurunan akurasi'); plt.title('Permutation importance (Gradient Boosting)')
plt.show()

## 4. Penjelasan model-agnostik: partial dependence
Bagaimana prediksi berubah ketika satu fitur diubah, fitur lain dirata-rata.

In [ ]:
top = list(imp.tail(2).index)
PartialDependenceDisplay.from_estimator(black, X_train, top, kind='average')
plt.tight_layout(); plt.show()

## 5. Model pengganti global (global surrogate)
Latih pohon sederhana untuk meniru **prediksi black box** (bukan label asli).

In [ ]:
pred_black_train = black.predict(X_train)
surrogate = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, pred_black_train)

# fidelity = seberapa mirip surrogate dengan black box di data test
pred_black_test = black.predict(X_test)
fidelity = accuracy_score(pred_black_test, surrogate.predict(X_test))
print('Fidelity surrogate terhadap black box:', round(fidelity, 4))
print('Akurasi surrogate terhadap label asli :', round(surrogate.score(X_test, y_test), 4))
print()
print(export_text(surrogate, feature_names=list(X.columns)))

## Catatan
- (trade-off akurasi vs interpretabilitas, kapan butuh model yang bisa dijelaskan, kelemahan surrogate, dll.)